# Week 73, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w073d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import re
names = ["Text_Kit", "text.kit", "TEXT-KIT", "textkit"]
print({re.sub(r"[-_.]+", "-", n).lower() for n in names})

**Predict before you run:** what will this print?



In [ ]:
import tomllib
d = tomllib.loads('[project]\nname = "a"\nversion = "1.0"\n[project.optional-dependencies]\ndev = ["pytest"]\n')
print(list(d["project"]), d["project"]["optional-dependencies"])

**Predict before you run:** what will this print?



In [ ]:
def t(v):
    return tuple(int(x) for x in v.split("."))
versions = ["2.0.1", "2.0.10", "2.0.9", "10.0"]
print(max(versions), max(versions, key=t))

**Predict before you run:** what will this print?



In [ ]:
filename = "zope_interface-6.4-cp312-cp312-manylinux_2_17_x86_64.whl"
name, version, py, abi, platform = filename[:-4].split("-")
print(name.replace("_", "-"), version, py == abi, platform.startswith("manylinux"))

**Predict before you run:** what will this print?



In [ ]:
import base64, hashlib
d = hashlib.sha256(b"").digest()
print(len(d), len(base64.urlsafe_b64encode(d)), len(base64.urlsafe_b64encode(d).rstrip(b"=")))

### Exercise: Pinned, ranged or open?

Write `pin_report(pyproject)`: for each entry of `project.dependencies` in the TOML text, classify it as `"pinned"` (a single `==` clause without `*`), `"open"` (no version clauses at all) or `"ranged"` (anything else), ignoring markers after `;`. Return a dict from the requirement's **normalised name** to its class, and `is_library_friendly(report)`: true if nothing is pinned.

In [ ]:
import re, tomllib

def pin_report(pyproject):
    ___

def is_library_friendly(report):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pins():\n    \"ranges for libraries\"\n    text = '[project]\\nname = \"x\"\\nversion = \"1\"\\ndependencies = [\"rich==13.7.1\", \"Click>=8\", \"idna\", \"attrs==23.*\", \"tomli; python_version < \\'3.11\\'\"]\\n'\n    report = pin_report(text)\n    assert report == {\"rich\": \"pinned\", \"click\": \"ranged\", \"idna\": \"open\", \"attrs\": \"ranged\", \"tomli\": \"open\"}\n    assert not is_library_friendly(report) and is_library_friendly({\"a\": \"open\", \"b\": \"ranged\"})\n    assert pin_report('[project]\\nname = \"x\"\\nversion = \"1\"\\n') == {}")

### Exercise: Is this upgrade safe?

Write `upgrade_risk(current, candidate)` for SemVer versions without pre-releases: `"none"` if the candidate isn't newer; `"breaking"` if the major number differs, or if both are `0.x` and the minor number differs; `"features"` if the minor number differs; otherwise `"fixes"`. Then `safe_upgrades(locked, available)`: for each locked package, the newest available version whose risk is `"fixes"` or `"features"` (or `None` if there isn't one), as a dict.

In [ ]:
def upgrade_risk(current, candidate):
    ___

def safe_upgrades(locked, available):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_risk():\n    \"SemVer's promise\"\n    assert upgrade_risk(\"1.4.2\", \"1.4.3\") == \"fixes\" and upgrade_risk(\"1.4.2\", \"1.6.0\") == \"features\"\n    assert upgrade_risk(\"1.4.2\", \"2.0.0\") == \"breaking\" and upgrade_risk(\"0.4.2\", \"0.5.0\") == \"breaking\"\n    assert upgrade_risk(\"0.4.2\", \"0.4.9\") == \"fixes\" and upgrade_risk(\"1.4.2\", \"1.4.2\") == \"none\" and upgrade_risk(\"1.4.2\", \"1.3.0\") == \"none\"\n    locked = {\"rich\": \"13.7.1\", \"attrs\": \"23.2.0\", \"idna\": \"3.7\"}\n    available = {\"rich\": [\"13.7.1\", \"13.9.4\", \"14.0.0\"], \"attrs\": [\"24.1.0\"], \"idna\": [\"3.7\", \"3.6\"]}\n    assert safe_upgrades(locked, available) == {\"rich\": \"13.9.4\", \"attrs\": None, \"idna\": None}")

### Exercise: Forge: a build backend

Write `Project` as described above. `record_hash` and `normalise` are given.

In [ ]:
import base64, csv, hashlib, io, re, tarfile, tomllib, zipfile


def record_hash(data):
    digest = hashlib.sha256(data).digest()
    return "sha256=" + base64.urlsafe_b64encode(digest).rstrip(b"=").decode()


def normalise(name):
    return re.sub(r"[-_.]+", "-", name).lower()


class Project:
    def __init__(self, files):
        self.files = dict(files)

    def metadata(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("FILES = {\n    \"pyproject.toml\": '[build-system]\\nrequires = [\"hatchling\"]\\nbuild-backend = \"hatchling.build\"\\n\\n[project]\\nname = \"Text-Kit\"\\nversion = \"0.3.0\"\\ndescription = \"Small text tools\"\\nreadme = \"README.md\"\\nrequires-python = \">=3.10\"\\ndependencies = [\"rich>=13\", \"tomli>=2; python_version < \\'3.11\\'\"]\\n\\n[project.optional-dependencies]\\npdf = [\"pypdf>=4\"]\\nfast = [\"regex; sys_platform != \\'win32\\'\"]\\n\\n[project.scripts]\\ntk = \"text_kit.cli:main\"\\n',\n    \"README.md\": \"# Text-Kit\\n\\nSmall text tools.\\n\",\n    \"src/text_kit/__init__.py\": \"from .core import shout\\n\",\n    \"src/text_kit/core.py\": \"def shout(s):\\n    return s.upper()\\n\",\n    \"src/text_kit/cli.py\": \"def main():\\n    return 0\\n\",\n    \"src/text_kit/data/words.txt\": \"alpha\\nbeta\\n\",\n    \"src/text_kit/__pycache__/core.cpython-312.pyc\": \"junk\",\n    \"tests/test_core.py\": \"def test_x():\\n    pass\\n\",\n}\n\ndef test_metadata_and_problems():\n    \"reading the project and finding problems\"\n    p = Project(FILES)\n    m = p.metadata()\n    assert m[\"name\"] == \"Text-Kit\" and m[\"version\"] == \"0.3.0\" and m[\"extras\"] == {\"pdf\": [\"pypdf>=4\"], \"fast\": [\"regex; sys_platform != 'win32'\"]}\n    assert p.problems() == []\n    bad = dict(FILES)\n    bad[\"pyproject.toml\"] = bad[\"pyproject.toml\"].replace('readme = \"README.md\"', 'readme = \"docs/README.md\"').replace('tk = \"text_kit.cli:main\"', 'tk = \"text_kit.cli\"')\n    del bad[\"src/text_kit/__init__.py\"]\n    assert Project(bad).problems() == [\"readme not found: docs/README.md\", \"package text_kit has no __init__.py\", \"bad script: tk\"]\n    assert Project({\"README.md\": \"\"}).problems() == [\"no pyproject.toml\"]\n    assert Project({\"pyproject.toml\": '[project]\\nname = \"x\"\\nversion = \"1\"\\n'}).problems() == [\"no package under src/\"]\n\ndef test_metadata_text():\n    \"METADATA, including extras\"\n    text = Project(FILES).metadata_text()\n    head, _, body = text.partition(\"\\n\\n\")\n    assert head.splitlines() == [\n        \"Metadata-Version: 2.3\", \"Name: Text-Kit\", \"Version: 0.3.0\", \"Summary: Small text tools\", \"Requires-Python: >=3.10\",\n        \"Requires-Dist: rich>=13\", \"Requires-Dist: tomli>=2; python_version < '3.11'\",\n        \"Provides-Extra: pdf\", 'Requires-Dist: pypdf>=4; extra == \"pdf\"',\n        \"Provides-Extra: fast\", \"Requires-Dist: regex; (sys_platform != 'win32') and extra == \\\"fast\\\"\",\n    ]\n    assert body == \"# Text-Kit\\n\\nSmall text tools.\\n\"\n\ndef test_wheel():\n    \"a complete wheel\"\n    import io, zipfile, csv\n    name, data = Project(FILES).wheel()\n    assert name == \"text_kit-0.3.0-py3-none-any.whl\"\n    z = zipfile.ZipFile(io.BytesIO(data))\n    info = \"text_kit-0.3.0.dist-info/\"\n    assert sorted(z.namelist()) == sorted([\"text_kit/__init__.py\", \"text_kit/cli.py\", \"text_kit/core.py\", \"text_kit/data/words.txt\",\n                                           info + \"METADATA\", info + \"WHEEL\", info + \"entry_points.txt\", info + \"RECORD\"])\n    assert z.read(\"text_kit/data/words.txt\") == b\"alpha\\nbeta\\n\" and \"tk = text_kit.cli:main\" in z.read(info + \"entry_points.txt\").decode()\n    rows = list(csv.reader(io.StringIO(z.read(info + \"RECORD\").decode())))\n    assert {r[0] for r in rows} == set(z.namelist())\n    assert all(r[1] == record_hash(z.read(r[0])) for r in rows if r[1])\n    broken = dict(FILES)\n    del broken[\"src/text_kit/__init__.py\"]\n    try:\n        Project(broken).wheel()\n        assert False\n    except ValueError as e:\n        assert \"no __init__.py\" in str(e)\n\ndef test_sdist():\n    \"a source distribution\"\n    import io, tarfile\n    name, data = Project(FILES).sdist()\n    assert name == \"text_kit-0.3.0.tar.gz\"\n    tar = tarfile.open(fileobj=io.BytesIO(data), mode=\"r:gz\")\n    names = tar.getnames()\n    assert \"text_kit-0.3.0/PKG-INFO\" in names and \"text_kit-0.3.0/pyproject.toml\" in names and \"text_kit-0.3.0/tests/test_core.py\" in names\n    assert tar.extractfile(\"text_kit-0.3.0/PKG-INFO\").read().decode().startswith(\"Metadata-Version: 2.3\\nName: Text-Kit\\n\")")

> **What a backend owes its users.** Notice how much of today was **checking**, not building. A wheel that builds from a broken project (a missing `__init__.py`, a script pointing at nothing, a README that isn't there) fails later, on someone else's machine, where it's hardest to diagnose. Real backends refuse early with a clear message for the same reason. And notice that the wheel contains nothing from `tests/`, while the sdist contains everything: the wheel is what users run, the sdist is what anyone needs to rebuild, test and audit it.

### Exercise: Forge bonus: checking a wheel against its source

Two classic packaging mistakes: a file in the source that never made it into the wheel (a data file the build configuration didn't include), and a wheel built from an older version of the project. Write `compare(files, wheel_bytes)`: compare the files under `src/` (with `src/` removed, skipping `__pycache__` and `.pyc`, as in the Forge) with the wheel's files outside its `.dist-info` folder, returning a dict with sorted lists `"missing"` (in the source, not the wheel), `"extra"` (in the wheel, not the source) and `"changed"` (in both, with different contents). Then `drift(pyproject, wheel_bytes)`: a list of messages comparing the project table with the wheel's `METADATA`: `"name: A in pyproject, B in wheel"` and `"version: A in pyproject, B in wheel"` if they differ, then `"not in wheel: DEP"` for each runtime dependency in the pyproject but not among the wheel's `Requires-Dist` lines that have no `extra` marker, and `"only in wheel: DEP"` for each such wheel requirement not in the pyproject (compare the requirement strings exactly, in the order they appear).

In [ ]:
import io, tomllib, zipfile
from email.parser import Parser

def compare(files, wheel_bytes):
    ___

def drift(pyproject, wheel_bytes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make_wheel(contents):\n    import io, zipfile\n    buf = io.BytesIO()\n    with zipfile.ZipFile(buf, \"w\") as z:\n        for p, d in contents.items():\n            z.writestr(p, d)\n    return buf.getvalue()\n\ndef test_compare():\n    \"missing, extra and changed files\"\n    files = {\"pyproject.toml\": \"\", \"src/kit/__init__.py\": \"X = 1\\n\", \"src/kit/data/words.txt\": \"a\\n\",\n             \"src/kit/core.py\": \"def f():\\n    return 2\\n\", \"src/kit/__pycache__/x.pyc\": \"junk\"}\n    wheel = make_wheel({\"kit/__init__.py\": \"X = 1\\n\", \"kit/core.py\": \"def f():\\n    return 1\\n\", \"kit/old.py\": \"\",\n                        \"kit-1.0.dist-info/METADATA\": \"Name: kit\\n\"})\n    assert compare(files, wheel) == {\"missing\": [\"kit/data/words.txt\"], \"extra\": [\"kit/old.py\"], \"changed\": [\"kit/core.py\"]}\n\ndef test_drift():\n    \"stale metadata\"\n    pyproject = '[project]\\nname = \"kit\"\\nversion = \"1.1.0\"\\ndependencies = [\"rich>=13\", \"idna\"]\\n'\n    meta = \"Metadata-Version: 2.3\\nName: kit\\nVersion: 1.0.0\\nRequires-Dist: rich>=13\\nRequires-Dist: click\\nRequires-Dist: pypdf; extra == \\\"pdf\\\"\\n\"\n    wheel = make_wheel({\"kit/__init__.py\": \"\", \"kit-1.0.0.dist-info/METADATA\": meta})\n    assert drift(pyproject, wheel) == [\"version: 1.1.0 in pyproject, 1.0.0 in wheel\", \"not in wheel: idna\", \"only in wheel: click\"]\n    same = make_wheel({\"kit-1.1.0.dist-info/METADATA\": \"Metadata-Version: 2.3\\nName: kit\\nVersion: 1.1.0\\nRequires-Dist: rich>=13\\nRequires-Dist: idna\\n\"})\n    assert drift(pyproject, same) == []")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a build backend

A build backend is the code that pip calls (through `[build-system]`) to turn a source tree into distributions. Real ones (hatchling, setuptools, flit) do more, but the core is what you'll build: read the project, check it, write the metadata, and pack the files.

A project is given as a dict of path to text, such as `{"pyproject.toml": "...", "README.md": "...", "src/text_kit/__init__.py": "...", ...}`. Write a class **`Project(files)`** with:

- **`metadata()`**: from `pyproject.toml`'s `[project]` table, a dict with `"name"`, `"version"`, `"summary"` (from `description`), `"requires_python"`, `"dependencies"` (a list), `"extras"` (a dict of extra to list), `"scripts"` (a dict) and `"readme"`, using `None`, `[]` or `{}` for missing fields. Raise `ValueError` if there's no `pyproject.toml`.
- **`problems()`**: a list of problems in this order: just `["no pyproject.toml"]` if it's missing; otherwise `"no name"` and `"no version"` if missing; `"readme not found: PATH"`; `"no package under src/"` if no file is inside a folder under `src/`; `"package NAME has no __init__.py"` for each top-level folder under `src/` (sorted) without one; and `"bad script: COMMAND"` for each script not of the form `module:function`.
- **`metadata_text()`**: the `METADATA` text: `Metadata-Version: 2.3`, `Name`, `Version`, then `Summary` and `Requires-Python` if set, then a `Requires-Dist` line per dependency, then for each extra (in order) `Provides-Extra: EXTRA` followed by its dependencies as `Requires-Dist: SPEC; extra == "EXTRA"`, or, if the dependency has its own marker, `Requires-Dist: SPEC; (MARKER) and extra == "EXTRA"`. End the headers with a newline; if the readme exists, add a blank line and the readme's text.
- **`wheel()`**: raise `ValueError` (with the problems joined by `"; "`) if there are problems; otherwise return `(filename, bytes)` for a `py3-none-any` wheel named from the normalised name with underscores. It contains every file under `src/` (with `src/` removed from the path) except anything in a `__pycache__` folder or ending in `.pyc`, then the `.dist-info` folder with `METADATA`, `WHEEL` (as in Day 3, `Generator: atlas`), `entry_points.txt` if there are scripts, and `RECORD` last.
- **`sdist()`**: the same check, then `(filename, bytes)` for a `.tar.gz` named `{stem}.tar.gz`, where `stem` is the wheel's `name-version`, containing every project file under `{stem}/`, plus `{stem}/PKG-INFO` holding the metadata text.

## Forge bonus: checking a wheel against its source

## Weekly quiz

## Week 73 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can lay out a project with the src layout, write packages with relative imports, run them with python -m, and read their data with importlib.resources. ([Day 1](w073d1.html))
- I can read and write pyproject.toml, fill in its build-system and project tables, and parse names and requirement strings. ([Day 2](w073d2.html))
- I can explain sdists and wheels, read wheel names and tags, and build and check a wheel with its RECORD. ([Day 3](w073d3.html))
- I can compare and bump versions (SemVer and PEP 440), deprecate gently, and write changelogs from commits. ([Day 4](w073d4.html))
- I can match specifiers, lock dependencies with hashes, and resolve them with backtracking. ([Day 5](w073d5.html))
- I built a working build backend and checked a wheel against its source. (Today)

Tomorrow is a **rest day**. Next week is **code quality and collaboration**: style and PEP 8 in depth, code review, documentation, continuous integration and pre-commit hooks, the habits that let a team share one codebase.